In [29]:
# Install PySpark if not already available
# !pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, DateType, LongType
)
from pyspark.sql.window import Window

import json
import os
from datetime import date

# Initialize Spark session
spark = SparkSession.builder \
    .appName("ELT_Pipeline") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 4.1.1


In [30]:
# Path to the raw source file — update this if the file is in a different location
INPUT_PATH      = "orders_data.json"
CUSTOMERS_PATH  = "customers_data.csv"
PRODUCTS_PATH   = "products_data.csv"

# Data lake layer paths
RAW_PATH       = "data_lake/raw/orders/"
QUARANTINE_PATH = "data_lake/quarantine/orders/"
CLEAN_PATH     = "data_lake/clean/orders/"
CURATED_PATH   = "data_lake/curated/orders/"
KPI_PATH       = "data_lake/curated/kpis/"

# Create data lake directories
for path in [RAW_PATH, QUARANTINE_PATH, CLEAN_PATH, CURATED_PATH, KPI_PATH]:
    os.makedirs(path, exist_ok=True)

# Valid reference values
VALID_STATUSES = {"Delivered", "Pending", "Cancelled", "Returned"}
VALID_PAYMENT  = {"COD", "UPI", "Card", "NetBanking"}
VALID_SOURCES  = {"web", "mobile", "affiliate", "store"}

In [31]:
# Read orders_data.json directly — place the file in the same directory as this notebook
# Spark reads newline-delimited JSON (one JSON object per line)
raw_df = spark.read \
    .option("mode", "PERMISSIVE") \
    .json(INPUT_PATH)

# Add ingestion metadata columns
raw_df = raw_df \
    .withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source_file", F.input_file_name())

print(f"Total raw records loaded: {raw_df.count()}")
raw_df.printSchema()
raw_df.show(5, truncate=False)

Total raw records loaded: 313
root
 |-- customer_id: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- price: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- source: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = false)
 |-- _source_file: string (nullable = false)

+-----------+--------+----------+--------+------------+--------------+-----+----------+--------+---------+--------------------------+-----------------------------------------------------------------------------------+
|customer_id|discount|order_date|order_id|order_status|payment_method|price|product_id|quantity|source   |_ingested_at              |_source_file                                                                       |
+-----------+--------+-

In [32]:
# Persist raw data to Bronze layer as Parquet
raw_df.write.mode("overwrite").parquet(RAW_PATH + "parquet/")
print(f"Bronze layer saved to: {RAW_PATH}parquet/")

Bronze layer saved to: data_lake/raw/orders/parquet/


In [33]:
# try_cast returns null for malformed values instead of throwing
# This safely handles records like BAD_RECORD, TEN, BAD_PRICE, etc.
typed_df = raw_df \
    .withColumn("order_id", F.expr("try_cast(order_id as BIGINT)")) \
    .withColumn("customer_id", F.expr("try_cast(customer_id as BIGINT)")) \
    .withColumn("product_id", F.expr("try_cast(product_id as BIGINT)")) \
    .withColumn("quantity", F.expr("try_cast(quantity as INT)")) \
    .withColumn("price", F.expr("try_cast(price as DOUBLE)")) \
    .withColumn("discount", F.expr("try_cast(discount as DOUBLE)"))

# Parse date — invalid formats (e.g. 32-13-2025) become null
typed_df = typed_df.withColumn(
    "order_date",
    F.expr("try_to_date(order_date, 'yyyy-MM-dd')")
)


In [34]:
# Define individual validation rules
flag_df = typed_df \
    .withColumn("flag_null_order_id",    F.col("order_id").isNull()) \
    .withColumn("flag_null_customer_id", F.col("customer_id").isNull()) \
    .withColumn("flag_null_product_id",  F.col("product_id").isNull()) \
    .withColumn("flag_invalid_quantity", F.col("quantity").isNull() | (F.col("quantity") <= 0)) \
    .withColumn("flag_invalid_price",    F.col("price").isNull() | (F.col("price") <= 0)) \
    .withColumn("flag_invalid_discount", F.col("discount").isNull() | (F.col("discount") < 0) | (F.col("discount") > 100)) \
    .withColumn("flag_invalid_date",     F.col("order_date").isNull()) \
    .withColumn("flag_invalid_status",   ~F.col("order_status").isin(list(VALID_STATUSES))) \
    .withColumn("flag_invalid_payment",  ~F.col("payment_method").isin(list(VALID_PAYMENT))) \
    .withColumn("flag_invalid_source",   ~F.col("source").isin(list(VALID_SOURCES)))

# Aggregate all flags into a single boolean
flag_cols = [c for c in flag_df.columns if c.startswith("flag_")]
bad_record_condition = F.lit(False)
for col in flag_cols:
    bad_record_condition = bad_record_condition | F.col(col)

flag_df = flag_df.withColumn("is_bad_record", bad_record_condition)

print("Record quality summary:")
flag_df.groupBy("is_bad_record").count().show()

Record quality summary:
+-------------+-----+
|is_bad_record|count|
+-------------+-----+
|         true|    3|
|        false|  310|
+-------------+-----+



In [35]:
# Quarantine — bad records kept for audit and reprocessing
quarantine_df = flag_df.filter(F.col("is_bad_record") == True)
quarantine_df.write.mode("overwrite").parquet(QUARANTINE_PATH)
print(f"Quarantined records: {quarantine_df.count()}")

# Drop flag columns before persisting clean data
drop_cols = flag_cols + ["is_bad_record", "_ingested_at", "_source_file"]
clean_df  = flag_df.filter(F.col("is_bad_record") == False).drop(*drop_cols)
print(f"Clean records: {clean_df.count()}")

Quarantined records: 3
Clean records: 310


In [36]:
# Keep first occurrence of each order_id, ranked by order_date
window_spec = Window.partitionBy("order_id").orderBy(F.col("order_date").asc())

deduped_df = clean_df \
    .withColumn("_rank", F.row_number().over(window_spec)) \
    .filter(F.col("_rank") == 1) \
    .drop("_rank")

print(f"Records before deduplication: {clean_df.count()}")
print(f"Records after deduplication:  {deduped_df.count()}")

Records before deduplication: 310
Records after deduplication:  300


In [37]:
# Persist silver layer
deduped_df.write.mode("overwrite").parquet(CLEAN_PATH)
print("Silver layer saved.")
deduped_df.show(5)

Silver layer saved.
+-----------+--------+----------+--------+------------+--------------+------+----------+--------+---------+
|customer_id|discount|order_date|order_id|order_status|payment_method| price|product_id|quantity|   source|
+-----------+--------+----------+--------+------------+--------------+------+----------+--------+---------+
|        109|    21.0|2025-01-27|       1|   Delivered|           COD|6466.0|      1265|      10|      web|
|        160|    20.0|2025-05-06|       2|   Delivered|           UPI| 741.0|      1104|      10|   mobile|
|        173|    16.0|2025-01-16|       3|     Pending|           COD|5470.0|      1034|       4|      web|
|        227|    21.0|2025-03-10|       4|     Pending|           COD|5451.0|      1038|      10|      web|
|        290|    30.0|2025-03-23|       5|   Delivered|           UPI|2436.0|      1116|       5|affiliate|
+-----------+--------+----------+--------+------------+--------------+------+----------+--------+---------+
only sho

In [38]:
# Read customer dimension from CSV
customer_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(CUSTOMERS_PATH)

# Cast customer_id to LongType to match orders
customer_df = customer_df.withColumn("customer_id", F.col("customer_id").cast(LongType()))

print(f"Customer dimension records: {customer_df.count()}")
customer_df.printSchema()
customer_df.show(5, truncate=False)

Customer dimension records: 301
root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: string (nullable = true)

+-----------+-------------------+---------------------------+----------+-------+-----+-----------+
|customer_id|name               |email                      |phone     |city   |state|signup_date|
+-----------+-------------------+---------------------------+----------+-------+-----+-----------+
|1          |Amanda Nguyen      |kellylucas@sullivan.biz    |7963870457|Chennai|TN   |2026-04-08 |
|2          |Matthew Stevens Jr.|alexandermoore@hotmail.com |9443560804|Pune   |MH   |2026-03-23 |
|3          |Doris Wilson       |thernandez@curry-sparks.com|0077792493|Delhi  |DL   |2025-11-09 |
|4          |Caitlin Cox        |swells@gmail.com           |4489711333|Mumbai |MH   |2025-12-10 |
|5

In [39]:
# Read product dimension from CSV
product_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(PRODUCTS_PATH)

# Cast product_id to LongType to match orders
product_df = product_df.withColumn("product_id", F.expr("try_cast (product_id as BIGINT)"))

print(f"Product dimension records: {product_df.count()}")
product_df.printSchema()
product_df.show(5, truncate=False)

Product dimension records: 289
root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: string (nullable = true)

+----------+------------+-----------+-------+-----+
|product_id|product_name|category   |brand  |price|
+----------+------------+-----------+-------+-----+
|1001      |Lite Laptop |Electronics|Sony   |6710 |
|1002      |Ultra Laptop|Electronics|Sony   |29580|
|1003      |Lite Laptop |Electronics|Sony   |84705|
|1004      |Ultra Laptop|Electronics|Samsung|65506|
|1005      |Ultra Laptop|Electronics|HP     |86155|
+----------+------------+-----------+-------+-----+
only showing top 5 rows


In [40]:
# Left join orders with customer and product dimensions
# Select and alias columns explicitly to avoid name collisions (e.g., name -> customer_name, product price -> product_price)
enriched_df = deduped_df.alias("o") \
    .join(customer_df.alias("c"), on="customer_id", how="left") \
    .join(product_df.alias("p"), on="product_id", how="left") \
    .select(
        F.col("o.*"),
        F.col("c.name").alias("customer_name"),
        F.col("c.city").alias("city"),
        F.col("c.state").alias("state"),
        F.col("c.signup_date").alias("signup_date"),
        F.col("p.product_name").alias("product_name"),
        F.col("p.category").alias("category"),
        F.col("p.brand").alias("brand"),
        F.col("p.price").alias("product_price")
    )

print(f"Enriched records: {enriched_df.count()}")
enriched_df.select(
    "order_id", "customer_name", "city",
    "product_name", "category", "order_status"
).show(5)

Enriched records: 300
+--------+----------------+---------+-----------------+-----------+------------+
|order_id|   customer_name|     city|     product_name|   category|order_status|
+--------+----------------+---------+-----------------+-----------+------------+
|       1|   Dwayne Crosby|Ahmedabad|Ultra AI Handbook|      Books|   Delivered|
|       2|    Tonya Malone|  Lucknow|      Pro Charger|Accessories|   Delivered|
|       3|Patricia Sellers|Hyderabad|      Max Monitor|Electronics|     Pending|
|       4| Scott Rodriguez|   Jaipur|     Ultra Tablet|Electronics|     Pending|
|       5|        John Key|  Kolkata|     Ultra Webcam|Accessories|   Delivered|
+--------+----------------+---------+-----------------+-----------+------------+
only showing top 5 rows


In [41]:
# Compute revenue metrics per order
curated_df = enriched_df \
    .withColumn(
        "gross_revenue",
        F.round(F.col("price") * F.col("quantity"), 2)
    ) \
    .withColumn(
        "discount_amount",
        F.round(F.col("gross_revenue") * F.col("discount") / 100, 2)
    ) \
    .withColumn(
        "net_revenue",
        F.round(F.col("gross_revenue") - F.col("discount_amount"), 2)
    ) \
    .withColumn("order_month", F.month(F.col("order_date"))) \
    .withColumn("order_year",  F.year(F.col("order_date"))) \
    .withColumn("order_quarter",
        F.concat(F.lit("Q"), F.ceil(F.month(F.col("order_date")) / 3).cast(StringType()))
    ) \
    .withColumn("is_high_value", F.col("net_revenue") > 10000)

curated_df.select(
    "order_id", "gross_revenue", "discount_amount", "net_revenue",
    "order_month", "order_quarter", "is_high_value"
).show(5)

+--------+-------------+---------------+-----------+-----------+-------------+-------------+
|order_id|gross_revenue|discount_amount|net_revenue|order_month|order_quarter|is_high_value|
+--------+-------------+---------------+-----------+-----------+-------------+-------------+
|       1|      64660.0|        13578.6|    51081.4|          1|           Q1|         true|
|       2|       7410.0|         1482.0|     5928.0|          5|           Q2|        false|
|       3|      21880.0|         3500.8|    18379.2|          1|           Q1|         true|
|       4|      54510.0|        11447.1|    43062.9|          3|           Q1|         true|
|       5|      12180.0|         3654.0|     8526.0|          3|           Q1|        false|
+--------+-------------+---------------+-----------+-----------+-------------+-------------+
only showing top 5 rows


In [42]:
# Compute per-customer lifetime net revenue across all their orders
customer_ltv = curated_df \
    .groupBy("customer_id") \
    .agg(F.sum("net_revenue").alias("customer_ltv"))

# Join LTV back to curated_df and assign tier based on spend thresholds
# Thresholds (adjust to match your data distribution):
#   Platinum : lifetime net revenue >= 100,000
#   Gold     : lifetime net revenue >= 50,000
#   Silver   : lifetime net revenue >= 20,000
#   Bronze   : everything else
curated_df = curated_df \
    .join(customer_ltv, on="customer_id", how="left") \
    .withColumn(
        "tier",
        F.when(F.col("customer_ltv") >= 100000, "Platinum")
         .when(F.col("customer_ltv") >= 50000,  "Gold")
         .when(F.col("customer_ltv") >= 20000,  "Silver")
         .otherwise("Bronze")
    ) \
    .drop("customer_ltv")

print("Tier distribution:")
curated_df.groupBy("tier").count().orderBy("tier").show()

curated_df.select(
    "order_id", "customer_name", "city", "tier",
    "product_name", "category", "order_status"
).show(5)

Tier distribution:
+--------+-----+
|    tier|count|
+--------+-----+
|  Bronze|   75|
|    Gold|   88|
|Platinum|   15|
|  Silver|  122|
+--------+-----+

+--------+----------------+---------+------+------------+-----------+------------+
|order_id|   customer_name|     city|  tier|product_name|   category|order_status|
+--------+----------------+---------+------+------------+-----------+------------+
|       6|    Melissa Hall|  Kolkata|Silver|     Max Bag|    Fashion|   Delivered|
|       4| Scott Rodriguez|   Jaipur|Silver|Ultra Tablet|Electronics|     Pending|
|       2|    Tonya Malone|  Lucknow|Silver| Pro Charger|Accessories|   Delivered|
|       5|        John Key|  Kolkata|  Gold|Ultra Webcam|Accessories|   Delivered|
|       3|Patricia Sellers|Hyderabad|  Gold| Max Monitor|Electronics|     Pending|
+--------+----------------+---------+------+------------+-----------+------------+
only showing top 5 rows


In [43]:
# Save curated (gold) layer
curated_df.write.mode("overwrite").partitionBy("order_year", "order_month").parquet(CURATED_PATH)
print("Gold layer saved.")

Gold layer saved.


In [44]:
overall_kpi = curated_df.agg(
    F.count("order_id").alias("total_orders"),
    F.round(F.sum("gross_revenue"), 2).alias("total_gross_revenue"),
    F.round(F.sum("discount_amount"), 2).alias("total_discount"),
    F.round(F.sum("net_revenue"), 2).alias("total_net_revenue"),
    F.round(F.avg("net_revenue"), 2).alias("avg_order_value"),
    F.round(F.avg("discount"), 2).alias("avg_discount_pct"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.countDistinct("product_id").alias("unique_products"),
)
print("Overall KPIs:")
overall_kpi.show(truncate=False)

Overall KPIs:
+------------+-------------------+--------------+-----------------+---------------+----------------+----------------+---------------+
|total_orders|total_gross_revenue|total_discount|total_net_revenue|avg_order_value|avg_discount_pct|unique_customers|unique_products|
+------------+-------------------+--------------+-----------------+---------------+----------------+----------------+---------------+
|300         |8925377.0          |1745677.81    |7179699.19       |23932.33       |19.76           |190             |180            |
+------------+-------------------+--------------+-----------------+---------------+----------------+----------------+---------------+



In [45]:
status_kpi = curated_df \
    .groupBy("order_status") \
    .agg(
        F.count("order_id").alias("order_count"),
        F.round(F.sum("net_revenue"), 2).alias("net_revenue"),
        F.round(F.avg("net_revenue"), 2).alias("avg_net_revenue"),
    ) \
    .orderBy(F.col("net_revenue").desc())

status_kpi.show()

+------------+-----------+-----------+---------------+
|order_status|order_count|net_revenue|avg_net_revenue|
+------------+-----------+-----------+---------------+
|    Returned|         87| 2282214.14|       26232.35|
|     Pending|         79|  1857557.8|       23513.39|
|   Cancelled|         60| 1551295.28|       25854.92|
|   Delivered|         74| 1488631.97|       20116.65|
+------------+-----------+-----------+---------------+



In [46]:
monthly_kpi = curated_df \
    .groupBy("order_year", "order_month") \
    .agg(
        F.count("order_id").alias("order_count"),
        F.round(F.sum("gross_revenue"), 2).alias("gross_revenue"),
        F.round(F.sum("net_revenue"), 2).alias("net_revenue"),
    ) \
    .orderBy("order_year", "order_month")

monthly_kpi.show()

+----------+-----------+-----------+-------------+-----------+
|order_year|order_month|order_count|gross_revenue|net_revenue|
+----------+-----------+-----------+-------------+-----------+
|      2025|          1|         63|    1967745.0| 1555463.86|
|      2025|          2|         51|    1131720.0|  919756.57|
|      2025|          3|         59|    1814620.0|  1445567.5|
|      2025|          4|         68|    2175641.0| 1782174.62|
|      2025|          5|         59|    1835651.0| 1476736.64|
+----------+-----------+-----------+-------------+-----------+



In [47]:
payment_kpi = curated_df \
    .groupBy("payment_method") \
    .agg(
        F.count("order_id").alias("order_count"),
        F.round(F.sum("net_revenue"), 2).alias("net_revenue"),
        F.round(
            F.count("order_id") / curated_df.count() * 100, 2
        ).alias("pct_orders")
    ) \
    .orderBy(F.col("net_revenue").desc())

payment_kpi.show()

+--------------+-----------+-----------+----------+
|payment_method|order_count|net_revenue|pct_orders|
+--------------+-----------+-----------+----------+
|           UPI|         84|  2115759.1|      28.0|
|           COD|         80| 1971761.05|     26.67|
|    NetBanking|         80| 1878898.24|     26.67|
|          Card|         56|  1213280.8|     18.67|
+--------------+-----------+-----------+----------+



In [48]:
channel_kpi = curated_df \
    .groupBy("source") \
    .agg(
        F.count("order_id").alias("order_count"),
        F.round(F.sum("net_revenue"), 2).alias("net_revenue"),
        F.round(F.avg("discount"), 2).alias("avg_discount_pct"),
    ) \
    .orderBy(F.col("net_revenue").desc())

channel_kpi.show()

+---------+-----------+-----------+----------------+
|   source|order_count|net_revenue|avg_discount_pct|
+---------+-----------+-----------+----------------+
|affiliate|         80| 2052128.23|           18.75|
|   mobile|         69| 1859011.11|           18.48|
|    store|         73| 1797931.95|           22.23|
|      web|         78|  1470627.9|           19.63|
+---------+-----------+-----------+----------------+



In [49]:
top_customers = curated_df \
    .groupBy("customer_id", "customer_name", "city", "tier") \
    .agg(
        F.count("order_id").alias("order_count"),
        F.round(F.sum("net_revenue"), 2).alias("lifetime_value"),
    ) \
    .orderBy(F.col("lifetime_value").desc()) \
    .limit(10)

top_customers.show(truncate=False)

+-----------+--------------+---------+--------+-----------+--------------+
|customer_id|customer_name |city     |tier    |order_count|lifetime_value|
+-----------+--------------+---------+--------+-----------+--------------+
|108        |Angel Fisher  |Mumbai   |Platinum|3          |197060.84     |
|134        |James Anderson|Kolkata  |Platinum|3          |117995.56     |
|93         |Gary Scott    |Jaipur   |Platinum|3          |105876.72     |
|236        |David Pollard |Hyderabad|Platinum|4          |104616.88     |
|288        |Jeffery Burch |Pune     |Platinum|2          |103731.4      |
|80         |Whitney Harvey|Ahmedabad|Gold    |2          |99452.15      |
|189        |Juan Brown    |Bangalore|Gold    |1          |94810.0       |
|119        |Pam Lopez     |Hyderabad|Gold    |3          |94196.18      |
|154        |Sarah Phillips|Bangalore|Gold    |3          |93792.6       |
|163        |Brian Sanford |Jaipur   |Gold    |2          |93668.86      |
+-----------+------------

In [50]:
top_products = curated_df \
    .groupBy("product_id", "product_name", "category", "brand") \
    .agg(
        F.count("order_id").alias("order_count"),
        F.sum("quantity").alias("total_units_sold"),
        F.round(F.sum("net_revenue"), 2).alias("net_revenue"),
    ) \
    .orderBy(F.col("net_revenue").desc()) \
    .limit(10)

top_products.show(truncate=False)

+----------+-----------------+---------------+-------+-----------+----------------+-----------+
|product_id|product_name     |category       |brand  |order_count|total_units_sold|net_revenue|
+----------+-----------------+---------------+-------+-----------+----------------+-----------+
|1265      |Ultra AI Handbook|Books          |Boat   |3          |25              |169271.8   |
|1275      |Ultra AI Handbook|Books          |Dell   |4          |26              |149082.67  |
|1287      |Pro ML Concepts  |Books          |Samsung|2          |15              |135841.0   |
|1018      |Lite Phone       |Electronics    |HP     |4          |26              |133163.48  |
|1152      |Lite AC          |Home Appliances|HP     |4          |24              |121117.26  |
|1255      |Lite Spark Guide |Books          |Sony   |3          |20              |110507.49  |
|1093      |Pro Headphones   |Accessories    |Boat   |4          |32              |108109.66  |
|1038      |Ultra Tablet     |Electronic

In [51]:
category_kpi = curated_df \
    .groupBy("category") \
    .agg(
        F.count("order_id").alias("order_count"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("net_revenue"), 2).alias("net_revenue"),
    ) \
    .orderBy(F.col("net_revenue").desc())

category_kpi.show()

+---------------+-----------+----------+-----------+
|       category|order_count|units_sold|net_revenue|
+---------------+-----------+----------+-----------+
|    Electronics|         66|       377| 1556059.18|
|Home Appliances|         67|       396| 1525835.98|
|        Fashion|         57|       319| 1423463.92|
|          Books|         54|       302| 1401507.98|
|    Accessories|         56|       326| 1272832.13|
+---------------+-----------+----------+-----------+



In [52]:
tier_kpi = curated_df \
    .filter(F.col("tier").isNotNull()) \
    .groupBy("tier") \
    .agg(
        F.countDistinct("customer_id").alias("unique_customers"),
        F.count("order_id").alias("order_count"),
        F.round(F.sum("net_revenue"), 2).alias("net_revenue"),
        F.round(F.avg("net_revenue"), 2).alias("avg_order_value"),
    ) \
    .orderBy(F.col("net_revenue").desc())

tier_kpi.show()

+--------+----------------+-----------+-----------+---------------+
|    tier|unique_customers|order_count|net_revenue|avg_order_value|
+--------+----------------+-----------+-----------+---------------+
|    Gold|              45|         88| 3239254.85|       36809.71|
|  Silver|              76|        122| 2695169.28|       22091.55|
|Platinum|               5|         15|   629281.4|       41952.09|
|  Bronze|              64|         75|  615993.66|        8213.25|
+--------+----------------+-----------+-----------+---------------+



In [53]:
total_orders = curated_df.count()

rate_kpi = curated_df \
    .groupBy("order_status") \
    .count() \
    .withColumn("rate_pct", F.round(F.col("count") / total_orders * 100, 2)) \
    .orderBy(F.col("count").desc())

rate_kpi.show()

+------------+-----+--------+
|order_status|count|rate_pct|
+------------+-----+--------+
|    Returned|   87|    29.0|
|     Pending|   79|   26.33|
|   Delivered|   74|   24.67|
|   Cancelled|   60|    20.0|
+------------+-----+--------+



In [54]:
# Write all KPI tables to Parquet
kpi_tables = {
    "overall":      overall_kpi,
    "by_status":    status_kpi,
    "by_month":     monthly_kpi,
    "by_payment":   payment_kpi,
    "by_channel":   channel_kpi,
    "top_customers":top_customers,
    "top_products": top_products,
    "by_category":  category_kpi,
    "by_tier":      tier_kpi,
    "order_rates":  rate_kpi,
}

for name, df in kpi_tables.items():
    df.write.mode("overwrite").parquet(KPI_PATH + name + "/")
    print(f"  KPI saved: {name}")

print("All KPIs persisted.")

  KPI saved: overall
  KPI saved: by_status
  KPI saved: by_month
  KPI saved: by_payment
  KPI saved: by_channel
  KPI saved: top_customers
  KPI saved: top_products
  KPI saved: by_category
  KPI saved: by_tier
  KPI saved: order_rates
All KPIs persisted.


In [55]:
raw_count        = raw_df.count()
bad_count        = quarantine_df.count()
clean_count      = clean_df.count()
deduped_count    = deduped_df.count()
duplicates_removed = clean_count - deduped_count

print("=" * 50)
print("ELT PIPELINE SUMMARY")
print("=" * 50)
print(f"  Raw records ingested      : {raw_count}")
print(f"  Bad records quarantined   : {bad_count}")
print(f"  Clean records             : {clean_count}")
print(f"  Duplicates removed        : {duplicates_removed}")
print(f"  Final curated records     : {deduped_count}")
print(f"  Data quality rate         : {round((deduped_count / raw_count) * 100, 1)}%")
print("=" * 50)

print("\nData lake paths:")
print(f"  Bronze  : {RAW_PATH}")
print(f"  Quarantine: {QUARANTINE_PATH}")
print(f"  Silver  : {CLEAN_PATH}")
print(f"  Gold    : {CURATED_PATH}")
print(f"  KPIs    : {KPI_PATH}")

ELT PIPELINE SUMMARY
  Raw records ingested      : 313
  Bad records quarantined   : 3
  Clean records             : 310
  Duplicates removed        : 10
  Final curated records     : 300
  Data quality rate         : 95.8%

Data lake paths:
  Bronze  : data_lake/raw/orders/
  Quarantine: data_lake/quarantine/orders/
  Silver  : data_lake/clean/orders/
  Gold    : data_lake/curated/orders/
  KPIs    : data_lake/curated/kpis/


In [56]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
